In [41]:
import pandas as pd

ci_taxonomies = "../data/c&I taxonomy/Final Business Rules based on XBRL C and I Taxonomy 2016 V1.3@06-08-2016/Final_Business_Rule_C&I_Taxonomy_2016_V1.3.xls"

In [42]:
df = pd.read_excel(ci_taxonomies)

In [43]:
df

,Sr. No.,Date,Tab,Element,New BR
0,1,2019-10-12,Specific rules for elements,CashAndCashEquivalents,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
1,2,2019-10-12,Mandatory Line Items,NumberOfEmployeesInTheCompanyAtTheEndOfTheFina...,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
2,3,2019-10-12,Specific rules for elements,WhetherCompanyHasPublishedSustainabilityReport...,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
3,4,2019-10-12,Specific rules for elements,NameOfMainProductOrService,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
4,5,2019-10-12,Specific rules for elements,DescriptionOfMainProductOrService,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
5,6,2019-10-12,Specific rules for elements,NICCodeOfProductOrService,"Mandatory if ""DateOfStartOfReportingPeriod"" is..."
6,7,2019-10-12,Mandaroty Line Item,DetailsOfMaterialContractsArrangementsTransact...,WhetherApprovalTakenFromBoardForMaterialContra...


In [ ]:
"""
a.py - Simple MCA C&I Taxonomy 2016 & Business Rules Parser
Target: ci_knowledge_base.json banane ke liye simple aur readable code.
"""

import json
import re
from pathlib import Path
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook

# -------------------------------------------------------------------------
# 1. PATHS SETUP (Auto-detects project root)
# -------------------------------------------------------------------------
def get_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'c&I taxonomy').exists():
            return p
    return Path('/home/vanshsharma/projects/xbrl_autopilot')

BASE_DIR = get_project_root()
TAXONOMY_DIR = BASE_DIR / 'data' / 'c&I taxonomy' / 'Final C and I Taxonomy 2016 V1.2@31-03-2016' / 'Taxonomy-2016-03-31'
RULES_EXCEL = BASE_DIR / 'data' / 'c&I taxonomy' / 'Final Business Rules based on XBRL C and I Taxonomy 2016 V1.3@06-08-2016' / 'Final_Business_Rule_C&I_Taxonomy_2016_V1.3.xls'
OUTPUT_JSON = BASE_DIR / 'ci_knowledge_base.json'

print(f'Project root detected: {BASE_DIR}')
print(f'Taxonomy exists: {TAXONOMY_DIR.exists()}')
print(f'Rules Excel exists: {RULES_EXCEL.exists()}')

# XML Namespaces
XS = "{http://www.w3.org/2001/XMLSchema}"
LINK = "{http://www.xbrl.org/2003/linkbase}"
XLINK = "{http://www.w3.org/1999/xlink}"
XBRLI = "{http://www.xbrl.org/2003/instance}"


# -------------------------------------------------------------------------
# 2. STEP 1: PARSE XSD SCHEMAS (CONCEPTS, TYPES, ROLES)
# -------------------------------------------------------------------------
def parse_schemas():
    print("[1/4] Parsing XSD schemas...")
    namespaces = {
        "in-ca": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-ca",
        "in-ca-roles": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-ca-roles",
        "in-ca-types": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-ca-types",
        "in-ci-ent": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-gaap/in-ci-ent",
        "in-gaap": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-gaap",
        "in-gaap-roles": "http://www.icai.org/xbrl/taxonomy/2016-03-31/in-gaap-roles",
    }
    
    concepts = {}
    types = {}
    roles = {}
    id_to_qname = {}

    for xsd_file in sorted(TAXONOMY_DIR.rglob("*.xsd")):
        rel_path = str(xsd_file.relative_to(TAXONOMY_DIR))
        tree = ET.parse(xsd_file)
        root = tree.getroot()
        tns = root.attrib.get("targetNamespace", "")

        # Prefix find karo (in-ca ya in-gaap etc.)
        prefix = next((p for p, uri in namespaces.items() if uri == tns), None)
        if not prefix:
            continue

        # A. Custom Types (like PANNumber, DINNumber, Enums)
        for ct in root.findall(f".//{XS}complexType"):
            name = ct.attrib.get("name")
            restr = ct.find(f".//{XS}restriction")
            if name and restr is not None:
                enums = [e.attrib.get("value") for e in restr.findall(f"{XS}enumeration")]
                pat = restr.find(f"{XS}pattern")
                length = restr.find(f"{XS}length")
                
                t_data = {"base": restr.attrib.get("base", "xbrli:stringItemType"), "source": rel_path}
                if enums: t_data["enumerations"] = enums
                if pat is not None: t_data["pattern"] = pat.attrib.get("value")
                if length is not None: t_data["length"] = length.attrib.get("value")
                types[f"{prefix}:{name}"] = t_data

        # B. Roles (ELRs like Balance Sheet, P&L, Notes)
        for rt in root.findall(f".//{LINK}roleType"):
            uri = rt.attrib.get("roleURI")
            def_el = rt.find(f"{LINK}definition")
            def_text = def_el.text.strip() if def_el is not None and def_el.text else ""
            m = re.match(r"^\[([0-9]{6}[a-z]?)\]", def_text)
            code = m.group(1) if m else uri.split("/")[-1]
            roles[uri] = {
                "id": rt.attrib.get("id"),
                "definition": def_text,
                "code": code,
                "source": rel_path,
            }

        # C. Concepts (Elements: Assets, Liabilities, Revenue, etc.)
        for elem in root.findall(f"{XS}element"):
            name = elem.attrib.get("name")
            if not name:
                continue
            qname = f"{prefix}:{name}"
            elem_id = elem.attrib.get("id")

            concepts[qname] = {
                "qname": qname,
                "name": name,
                "prefix": prefix,
                "namespace": tns,
                "type": elem.attrib.get("type"),
                "periodType": elem.attrib.get(f"{XBRLI}periodType"),
                "balance": elem.attrib.get(f"{XBRLI}balance"),
                "abstract": elem.attrib.get("abstract") == "true",
                "nillable": elem.attrib.get("nillable") == "true",
                "source": rel_path,
                "labels": {},
                "references": [],
            }
            if elem_id:
                id_to_qname[f"{xsd_file.name}#{elem_id}"] = qname
                id_to_qname[elem_id] = qname

    print(f"  -> {len(concepts)} concepts, {len(types)} types, {len(roles)} roles found.")
    return namespaces, concepts, types, roles, id_to_qname


# -------------------------------------------------------------------------
# 3. STEP 2: PARSE LABELS (EN + HI) & REFERENCES
# -------------------------------------------------------------------------
def parse_labels_and_references(concepts, id_to_qname):
    print("[2/4] Parsing English & Hindi labels, and MCA references...")
    
    # Label files
    for lab_file in sorted(TAXONOMY_DIR.rglob("*-lab-*.xml")):
        tree = ET.parse(lab_file)
        root = tree.getroot()
        
        # Locators map: xlink:label -> concept QName
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            href = loc.attrib.get(f"{XLINK}href", "")
            frag = href.split("#")[-1]
            qn = id_to_qname.get(frag)
            if qn:
                locs[lbl] = qn

        # Resource map: xlink:label -> label text & lang
        resources = {}
        for r in root.findall(f".//{LINK}label"):
            lbl = r.attrib.get(f"{XLINK}label")
            lang = r.attrib.get("{http://www.w3.org/XML/1998/namespace}lang") or r.attrib.get("xml:lang") or "en"
            role = (r.attrib.get(f"{XLINK}role") or "").split("/")[-1] or "label"
            resources[lbl] = (r.text.strip() if r.text else "", lang, role)

        # Connect using labelArc
        for arc in root.findall(f".//{LINK}labelArc"):
            from_lbl = arc.attrib.get(f"{XLINK}from")
            to_lbl = arc.attrib.get(f"{XLINK}to")
            qn = locs.get(from_lbl)
            if qn and to_lbl in resources and qn in concepts:
                text, lang, role = resources[to_lbl]
                if lang == "hi":
                    concepts[qn]["labels"]["label_hi"] = text
                else:
                    concepts[qn]["labels"][role] = text

    # Reference files
    for ref_file in sorted(TAXONOMY_DIR.rglob("*-ref-*.xml")):
        tree = ET.parse(ref_file)
        root = tree.getroot()
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
            qn = id_to_qname.get(frag)
            if qn:
                locs[lbl] = qn

        # Resource map: xlink:label -> reference dict
        ref_resources = {}
        for r in root.findall(f".//{LINK}reference"):
            lbl = r.attrib.get(f"{XLINK}label")
            parts = {child.tag.split("}")[-1]: child.text.strip() for child in r if child.text}
            if lbl and parts:
                ref_resources[lbl] = parts

        for arc in root.findall(f".//{LINK}referenceArc"):
            qn = locs.get(arc.attrib.get(f"{XLINK}from"))
            to_lbl = arc.attrib.get(f"{XLINK}to")
            if qn and to_lbl in ref_resources and qn in concepts:
                concepts[qn]["references"].append(ref_resources[to_lbl])

    # Add fallback labels for typed domain containers (MCA does not provide labels in XML for these)
    for qn, c in concepts.items():
        if not c["labels"].get("label"):
            if c["name"].endswith("Domain"):
                c["isTypedDomain"] = True
                readable = re.sub(r'(?<!^)(?=[A-Z])', ' ', c["name"])
                if readable.lower().endswith('domain'):
                    readable = re.sub(r'\s*domain$', '', readable, flags=re.I) + ' [Domain]'
                c["labels"]["label"] = readable

    print("  -> Labels and references merged into concepts.")


# -------------------------------------------------------------------------
# 4. STEP 3: PARSE LINKBASES (PRE, CAL, DEF RELATIONSHIPS)
# -------------------------------------------------------------------------
def parse_relationships(id_to_qname):
    print("[3/4] Parsing PRE, CAL, DEF linkbases...")
    presentation = {}
    calculation = {}
    definition = {}

    for xml_file in sorted(TAXONOMY_DIR.rglob("*.xml")):
        if any(k in str(xml_file) for k in ["-lab-", "-ref-", "META-INF"]):
            continue

        tree = ET.parse(xml_file)
        root = tree.getroot()

        for link in root:
            elr = link.attrib.get(f"{XLINK}role")
            if not elr:
                continue

            # Map locators
            locs = {}
            for loc in link.findall(f"{LINK}loc"):
                lbl = loc.attrib.get(f"{XLINK}label")
                frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
                qn = id_to_qname.get(frag)
                if qn:
                    locs[lbl] = qn

            # A. Presentation Link (Hierarchies)
            for arc in link.findall(f"{LINK}presentationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    order = float(arc.attrib.get("order", "1.0"))
                    presentation.setdefault(elr, []).append({
                        "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

            # B. Calculation Link (Summations & Weights)
            for arc in link.findall(f"{LINK}calculationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    w = float(arc.attrib.get("weight", "1.0"))
                    order = float(arc.attrib.get("order", "1.0"))
                    calculation.setdefault(elr, []).append({
                        "from": f, "to": t, "weight": int(w) if w.is_integer() else w, "order": int(order) if order.is_integer() else order
                    })

            # C. Definition Link (Dimensions & Tables)
            for arc in link.findall(f"{LINK}definitionArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    arcrole = arc.attrib.get(f"{XLINK}arcrole", "").split("/")[-1]
                    order = float(arc.attrib.get("order", "1.0"))
                    definition.setdefault(elr, []).append({
                        "arcrole": arcrole, "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

    # Sort arcs by order
    for d in [presentation, calculation, definition]:
        for k in d:
            d[k].sort(key=lambda x: (x["from"], x["order"]))

    print(f"  -> PRE roles: {len(presentation)}, CAL roles: {len(calculation)}, DEF roles: {len(definition)}.")
    return presentation, calculation, definition


# -------------------------------------------------------------------------
# 5. STEP 4: PARSE BUSINESS RULES (MANDATORY & NULL CHECKS)
# -------------------------------------------------------------------------
def parse_business_rules(concepts):
    print("[4/4] Parsing Business Rules from Excel (mandatory/null checks)...")
    business_rules = {
        "mandatoryConcepts": [],      # Globally mandatory elements (cannot be null)
        "mandatoryConsolidated": [],  # Mandatory in Consolidated reports
        "mandatoryLineItems": {},     # Table -> List of mandatory fields
        "rules": [],
    }

    if not RULES_EXCEL.exists():
        print(f"  ! Warning: Business rules excel not found at {RULES_EXCEL}")
        return business_rules

    wb = CalamineWorkbook.from_path(RULES_EXCEL)
    sheet_names = wb.sheet_names

    # Concept name lookup helper
    by_name = {c["name"].lower(): qn for qn, c in concepts.items()}

    # A. Specific rules for elements (Mandatory Null checks)
    if "Specific rules for elements" in sheet_names:
        rows = wb.get_sheet_by_name("Specific rules for elements").to_python()
        cur_code = None
        for r in rows:
            c0 = str(r[0] or "").strip()
            c1 = str(r[1] or "").strip()
            if c0.startswith("[") and "]" in c0:
                cur_code = re.match(r"^\[([0-9]{6}[a-z]?)\]", c0).group(1)
                continue
            if not c1 or c0 == "ELR/ Element Name":
                continue

            qn = by_name.get(c0.lower())
            rule_text = c1.lower()

            if "mandatory in case of consolidated" in rule_text:
                if qn and qn not in business_rules["mandatoryConsolidated"]:
                    business_rules["mandatoryConsolidated"].append(qn)
                    concepts[qn]["isMandatoryConsolidated"] = True
                rule_type = "mandatory-consolidated"
            elif "is a mandatory field" in rule_text or "this is a mandatory field" in rule_text or "mandatory field" in rule_text:
                if qn and qn not in business_rules["mandatoryConcepts"]:
                    business_rules["mandatoryConcepts"].append(qn)
                    concepts[qn]["isMandatory"] = True
                rule_type = "mandatory"
            else:
                rule_type = "validation-rule"

            business_rules["rules"].append({
                "element": c0,
                "concept": qn,
                "elr": cur_code,
                "type": rule_type,
                "ruleText": c1
            })

    # B. Mandatory Line Items for Tables (Table row null checks)
    ml_sheet = next((s for s in sheet_names if "mandatory line" in s.lower()), None)
    if ml_sheet:
        ml_rows = wb.get_sheet_by_name(ml_sheet).to_python()
        for r in ml_rows[3:]:
            if len(r) >= 3 and r[0] is not None:
                tbl_raw = str(r[1] or "").strip()
                items_raw = str(r[2] or "").strip()
                
                # Element names extract karo
                mand_qns = []
                for item in re.split(r"[,\n]+|\sand\s", items_raw):
                    clean_item = re.sub(r"^(Mandatory elements are\s*-\s*|Elements mandatory are\s*-\s*|Element for\s*)", "", item, flags=re.I).strip().strip("'\"")
                    clean_item = re.sub(r"\(closing\)", "", clean_item, flags=re.I).strip()
                    cqn = by_name.get(clean_item.lower())
                    if cqn:
                        mand_qns.append(cqn)

                if mand_qns:
                    business_rules["mandatoryLineItems"][tbl_raw] = mand_qns

    print(f"  -> {len(business_rules['mandatoryConcepts'])} globally mandatory concepts found (cannot be null).")
    print(f"  -> {len(business_rules['mandatoryLineItems'])} tables with mandatory line-items extracted.")
    return business_rules


# -------------------------------------------------------------------------
# 6. MAIN COMPILER: RUN ALL & EXPORT JSON
# -------------------------------------------------------------------------
def main():
    print("=" * 60)
    print("Starting C&I Taxonomy 2016 Extraction Pipeline")
    print("=" * 60)

    # 1. Schemas
    namespaces, concepts, types, roles, id_to_qname = parse_schemas()

    # 2. Labels & References
    parse_labels_and_references(concepts, id_to_qname)

    # 3. Relationships
    presentation, calculation, definition = parse_relationships(id_to_qname)

    # 4. Business Rules
    business_rules = parse_business_rules(concepts)

    # Assemble JSON
    canonical_kb = {
        "meta": {
            "name": "MCA C&I Taxonomy 2016 authority model",
            "taxonomyVersion": "Indian C&I General Purpose Taxonomy 2016-03-31 Ver 1.0",
            "entryPoint": "in-ci-ent-2016-03-31.xsd",
            "schemaRef": "https://www.mca.gov.in/V3XBRL/2016/07/26/Taxonomy/CnI/in-ci-ent-2016-03-31.xsd",
            "totalConcepts": len(concepts),
            "mandatoryConceptsCount": len(business_rules["mandatoryConcepts"]),
        },
        "namespaces": namespaces,
        "types": types,
        "roles": roles,
        "concepts": concepts,
        "presentation": presentation,
        "calculation": calculation,
        "definition": definition,
        "businessRules": business_rules,
    }

    # Save to file
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(canonical_kb, f, indent=2, ensure_ascii=False)

    print("=" * 60)
    print(f"DONE! File generated: {OUTPUT_JSON}")
    print(f"Size: {OUTPUT_JSON.stat().st_size / (1024*1024):.2f} MB")
    print("=" * 60)



Project root detected: /home/vanshsharma/projects/xbrl_autopilot
Taxonomy exists: True
Rules Excel exists: True


In [45]:
main()

Starting C&I Taxonomy 2016 Extraction Pipeline
[1/4] Parsing XSD schemas...
  -> 3660 concepts, 35 types, 130 roles found.
[2/4] Parsing English & Hindi labels, and MCA references...
  -> Labels and references merged into concepts.
[3/4] Parsing PRE, CAL, DEF linkbases...
  -> PRE roles: 47, CAL roles: 26, DEF roles: 112.
[4/4] Parsing Business Rules from Excel (mandatory/null checks)...
  -> 152 globally mandatory concepts found (cannot be null).
  -> 36 tables with mandatory line-items extracted.
DONE! File generated: /home/vanshsharma/projects/xbrl_autopilot/ci_knowledge_base.json
Size: 4.81 MB


In [46]:
"""
b.py - Simple MCA Costing Taxonomy 2019 & Business Rules Parser
Target: cost_knowledge_base.json banane ke liye simple aur readable code.
"""

import json
import re
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook

# -------------------------------------------------------------------------
# 1. PATHS SETUP (Auto-detects project root from any folder or notebook)
# -------------------------------------------------------------------------
def get_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "Costing Taxonomy").exists():
            return p
    try:
        f = Path(__file__).resolve()
        for p in [f.parent, *f.parents]:
            if (p / "data" / "Costing Taxonomy").exists():
                return p
    except NameError:
        pass
    return Path("/home/vanshsharma/projects/xbrl_autopilot")

BASE_DIR = get_project_root()
COST_ROOT = BASE_DIR / "data" / "Costing Taxonomy"
TAXONOMY_DIR = COST_ROOT / "Costing Taxonomy 2019 Version 1.1@31-03-2019" / "FinalCostingTaxonomy2019_V1.1" / "Taxonomy"
RULES_EXCEL = COST_ROOT / "Business rule based on Costing taxonomy 2019 V1.1@06-10-2019" / "Final_BusinessRules_based_on_CostingTaxonomy2019_V1.1.xlsx"
OUTPUT_JSON = BASE_DIR / "cost_knowledge_base.json"

# Auto unzip agar taxonomy folder unzipped nahi hai
if not TAXONOMY_DIR.exists():
    zip_path = COST_ROOT / "Costing Taxonomy 2019 Version 1.1@31-03-2019" / "FinalCostingTaxonomy2019_V1.1" / "Costing Taxonomy 2019.zip"
    if zip_path.exists():
        print(f"Extracting {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(TAXONOMY_DIR)

print(f"Project root detected: {BASE_DIR}")
print(f"Taxonomy directory exists: {TAXONOMY_DIR.exists()}")
print(f"Rules Excel exists: {RULES_EXCEL.exists()}")

# XML Namespaces
XS = "{http://www.w3.org/2001/XMLSchema}"
LINK = "{http://www.xbrl.org/2003/linkbase}"
XLINK = "{http://www.w3.org/1999/xlink}"
XBRLI = "{http://www.xbrl.org/2003/instance}"


# -------------------------------------------------------------------------
# 2. STEP 1: PARSE XSD SCHEMAS (CONCEPTS, TYPES, ROLES)
# -------------------------------------------------------------------------
def parse_schemas():
    print("[1/4] Parsing XSD schemas...")
    namespaces = {
        "in-cost": "http://www.icmai.in/xbrl/2018-04-01/in-cost",
        "in-cost-type": "http://www.icmai.in/xbrl/2018-04-01/in-cost-type",
        "in-cost-aud": "http://www.icmai.in/xbrl/2018-04-01/in-cost-aud",
        "in-cost-aud-role": "http://www.icmai.in/xbrl/2018-04-01/in-cost-aud-role",
    }
    
    concepts = {}
    types = {}
    roles = {}
    id_to_qname = {}

    for xsd_file in sorted(TAXONOMY_DIR.rglob("*.xsd")):
        rel_path = str(xsd_file.relative_to(TAXONOMY_DIR))
        tree = ET.parse(xsd_file)
        root = tree.getroot()
        tns = root.attrib.get("targetNamespace", "")

        prefix = next((p for p, uri in namespaces.items() if uri == tns), None)
        if not prefix:
            continue

        # A. Custom Types
        for ct in root.findall(f".//{XS}complexType") + root.findall(f".//{XS}simpleType"):
            name = ct.attrib.get("name")
            restr = ct.find(f".//{XS}restriction")
            if name and restr is not None:
                enums = [e.attrib.get("value") for e in restr.findall(f"{XS}enumeration")]
                pat = restr.find(f"{XS}pattern")
                length = restr.find(f"{XS}length")
                
                t_data = {"base": restr.attrib.get("base", "xbrli:stringItemType"), "source": rel_path}
                if enums: t_data["enumerations"] = enums
                if pat is not None: t_data["pattern"] = pat.attrib.get("value")
                if length is not None: t_data["length"] = length.attrib.get("value")
                types[f"{prefix}:{name}"] = t_data

        # B. Roles (ELRs like Cost Audit Report, General Information)
        for rt in root.findall(f".//{LINK}roleType"):
            uri = rt.attrib.get("roleURI")
            def_el = rt.find(f"{LINK}definition")
            def_text = def_el.text.strip() if def_el is not None and def_el.text else ""
            m = re.match(r"^\[([0-9]{6}[a-z]?)\]", def_text)
            code = m.group(1) if m else uri.split("/")[-1]
            roles[uri] = {
                "id": rt.attrib.get("id"),
                "definition": def_text,
                "code": code,
                "source": rel_path,
            }

        # C. Concepts
        for elem in root.findall(f"{XS}element"):
            name = elem.attrib.get("name")
            if not name:
                continue
            qname = f"{prefix}:{name}"
            elem_id = elem.attrib.get("id")

            concepts[qname] = {
                "qname": qname,
                "name": name,
                "prefix": prefix,
                "namespace": tns,
                "type": elem.attrib.get("type"),
                "periodType": elem.attrib.get(f"{XBRLI}periodType"),
                "balance": elem.attrib.get(f"{XBRLI}balance"),
                "abstract": elem.attrib.get("abstract") == "true",
                "nillable": elem.attrib.get("nillable") == "true",
                "source": rel_path,
                "labels": {},
                "references": [],
            }
            if elem_id:
                id_to_qname[f"{xsd_file.name}#{elem_id}"] = qname
                id_to_qname[elem_id] = qname

    print(f"  -> {len(concepts)} concepts, {len(types)} types, {len(roles)} roles found.")
    return namespaces, concepts, types, roles, id_to_qname


# -------------------------------------------------------------------------
# 3. STEP 2: PARSE LABELS (EN + HI) & REFERENCES
# -------------------------------------------------------------------------
def parse_labels_and_references(concepts, id_to_qname):
    print("[2/4] Parsing English & Hindi labels, and Cost references...")
    
    # Label files
    for lab_file in sorted(TAXONOMY_DIR.rglob("*-lab-*.xml")):
        tree = ET.parse(lab_file)
        root = tree.getroot()
        
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            href = loc.attrib.get(f"{XLINK}href", "")
            frag = href.split("#")[-1]
            qn = id_to_qname.get(frag)
            if not qn and "_" in frag:
                p, _, n = frag.partition("_")
                if f"{p}:{n}" in concepts:
                    qn = f"{p}:{n}"
            if qn:
                locs[lbl] = qn

        resources = {}
        for r in root.findall(f".//{LINK}label"):
            lbl = r.attrib.get(f"{XLINK}label")
            lang = r.attrib.get("{http://www.w3.org/XML/1998/namespace}lang") or r.attrib.get("xml:lang") or "en"
            role = (r.attrib.get(f"{XLINK}role") or "").split("/")[-1] or "label"
            resources[lbl] = (r.text.strip() if r.text else "", lang, role)

        for arc in root.findall(f".//{LINK}labelArc"):
            from_lbl = arc.attrib.get(f"{XLINK}from")
            to_lbl = arc.attrib.get(f"{XLINK}to")
            qn = locs.get(from_lbl)
            if qn and to_lbl in resources and qn in concepts:
                text, lang, role = resources[to_lbl]
                if lang == "hi":
                    concepts[qn]["labels"]["label_hi"] = text
                else:
                    concepts[qn]["labels"][role] = text

    # Reference files
    for ref_file in sorted(TAXONOMY_DIR.rglob("*-ref-*.xml")):
        tree = ET.parse(ref_file)
        root = tree.getroot()
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
            qn = id_to_qname.get(frag)
            if not qn and "_" in frag:
                p, _, n = frag.partition("_")
                if f"{p}:{n}" in concepts:
                    qn = f"{p}:{n}"
            if qn:
                locs[lbl] = qn

        ref_resources = {}
        for r in root.findall(f".//{LINK}reference"):
            lbl = r.attrib.get(f"{XLINK}label")
            parts = {child.tag.split("}")[-1]: child.text.strip() for child in r if child.text}
            if lbl and parts:
                ref_resources[lbl] = parts

        for arc in root.findall(f".//{LINK}referenceArc"):
            qn = locs.get(arc.attrib.get(f"{XLINK}from"))
            to_lbl = arc.attrib.get(f"{XLINK}to")
            if qn and to_lbl in ref_resources and qn in concepts:
                concepts[qn]["references"].append(ref_resources[to_lbl])

    print("  -> Labels and references merged into concepts.")


# -------------------------------------------------------------------------
# 4. STEP 3: PARSE LINKBASES (PRE, CAL, DEF RELATIONSHIPS)
# -------------------------------------------------------------------------
def parse_relationships(concepts, id_to_qname):
    print("[3/4] Parsing PRE, CAL, DEF linkbases...")
    presentation = {}
    calculation = {}
    definition = {}

    def resolve_qn(frag):
        qn = id_to_qname.get(frag)
        if not qn and "_" in frag:
            p, _, n = frag.partition("_")
            if f"{p}:{n}" in concepts:
                qn = f"{p}:{n}"
        return qn

    for xml_file in sorted(TAXONOMY_DIR.rglob("*.xml")):
        if any(k in str(xml_file) for k in ["-lab-", "-ref-", "META-INF"]):
            continue

        tree = ET.parse(xml_file)
        root = tree.getroot()

        for link in root:
            elr = link.attrib.get(f"{XLINK}role")
            if not elr:
                continue

            locs = {}
            for loc in link.findall(f"{LINK}loc"):
                lbl = loc.attrib.get(f"{XLINK}label")
                frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
                qn = resolve_qn(frag)
                if qn:
                    locs[lbl] = qn

            # A. Presentation Link
            for arc in link.findall(f"{LINK}presentationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    order = float(arc.attrib.get("order", "1.0"))
                    presentation.setdefault(elr, []).append({
                        "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

            # B. Calculation Link
            for arc in link.findall(f"{LINK}calculationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    w = float(arc.attrib.get("weight", "1.0"))
                    order = float(arc.attrib.get("order", "1.0"))
                    calculation.setdefault(elr, []).append({
                        "from": f, "to": t, "weight": int(w) if w.is_integer() else w, "order": int(order) if order.is_integer() else order
                    })

            # C. Definition Link
            for arc in link.findall(f"{LINK}definitionArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    arcrole = arc.attrib.get(f"{XLINK}arcrole", "").split("/")[-1]
                    order = float(arc.attrib.get("order", "1.0"))
                    definition.setdefault(elr, []).append({
                        "arcrole": arcrole, "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

    for d in [presentation, calculation, definition]:
        for k in d:
            d[k].sort(key=lambda x: (x["from"], x["order"]))

    print(f"  -> PRE roles: {len(presentation)}, CAL roles: {len(calculation)}, DEF roles: {len(definition)}.")
    return presentation, calculation, definition


# -------------------------------------------------------------------------
# 5. STEP 4: PARSE BUSINESS RULES (MANDATORY & NULL CHECKS)
# -------------------------------------------------------------------------
def parse_business_rules(concepts):
    print("[4/4] Parsing Business Rules from Excel (mandatory/null checks)...")
    business_rules = {
        "mandatoryConcepts": [],      # Cannot be null
        "conditionalMandatory": [],   # Mandatory on condition (e.g. date check)
        "mandatoryLineItems": {},     # Table -> List of mandatory fields
        "genericRules": [],
        "rules": [],
    }

    if not RULES_EXCEL.exists():
        print(f"  ! Warning: Business rules excel not found at {RULES_EXCEL}")
        return business_rules

    wb = CalamineWorkbook.from_path(RULES_EXCEL)
    sheet_names = wb.sheet_names

    by_name = {c["name"].lower(): qn for qn, c in concepts.items()}

    # A. Specific rules-CAR (Mandatory Null checks)
    spec_sheet = next((s for s in sheet_names if "specific rules" in s.lower()), None)
    if spec_sheet:
        rows = wb.get_sheet_by_name(spec_sheet).to_python()
        cur_elr = None
        for r in rows[1:]:
            c0 = str(r[0] or "").strip()
            c1 = str(r[1] or "").strip()
            c2 = str(r[2] or "").strip()
            c3 = str(r[3] or "").strip() if len(r) > 3 else ""

            if c0 == "ELR":
                m = re.search(r"\[([0-9]{6}[a-z]?)\]", c1)
                cur_elr = m.group(1) if m else c1
                continue
            if not c2 or c2 == "Element Name" or not c3:
                continue

            qn = by_name.get(c2.lower())
            rule_lower = c3.lower()

            if "mandatory" in rule_lower and not rule_lower.startswith("mandatory if"):
                if qn and qn not in business_rules["mandatoryConcepts"]:
                    business_rules["mandatoryConcepts"].append(qn)
                    concepts[qn]["isMandatory"] = True
                rule_type = "mandatory"
            elif "mandatory if" in rule_lower:
                if qn and qn not in business_rules["conditionalMandatory"]:
                    business_rules["conditionalMandatory"].append(qn)
                    concepts[qn]["isMandatoryConditional"] = True
                rule_type = "conditional-mandatory"
            else:
                rule_type = "validation-rule"

            business_rules["rules"].append({
                "element": c2,
                "concept": qn,
                "elr": cur_elr,
                "type": rule_type,
                "ruleText": c3
            })

    # B. Mandatory line item CAR
    ml_sheet = next((s for s in sheet_names if "mandatory line item" in s.lower()), None)
    if ml_sheet:
        ml_rows = wb.get_sheet_by_name(ml_sheet).to_python()
        cur_tbl = None
        for r in ml_rows[2:]:
            if len(r) < 3: continue
            c1 = str(r[1] or "").strip()
            c2 = str(r[2] or "").strip()
            if c1 and c1 != "Table name":
                cur_tbl = c1
            if cur_tbl and c2 and c2 != "Mandatory line-items":
                cqn = by_name.get(c2.lower())
                business_rules["mandatoryLineItems"].setdefault(cur_tbl, []).append(cqn if cqn else c2)

    # C. Generic Business rules
    gen_sheet = next((s for s in sheet_names if "generic" in s.lower()), None)
    if gen_sheet:
        gen_rows = wb.get_sheet_by_name(gen_sheet).to_python()
        for r in gen_rows:
            if len(r) >= 2 and r[0] is not None and str(r[0]).strip().replace(".0", "").isdigit():
                business_rules["genericRules"].append({
                    "sno": int(float(r[0])),
                    "description": str(r[1]).strip()
                })

    print(f"  -> {len(business_rules['mandatoryConcepts'])} mandatory concepts found (cannot be null).")
    print(f"  -> {len(business_rules['mandatoryLineItems'])} tables with mandatory line-items extracted.")
    return business_rules


# -------------------------------------------------------------------------
# 6. MAIN COMPILER: RUN ALL & EXPORT JSON
# -------------------------------------------------------------------------
def main():
    print("=" * 60)
    print("Starting MCA Costing Taxonomy 2019 Extraction Pipeline")
    print("=" * 60)

    # 1. Schemas
    namespaces, concepts, types, roles, id_to_qname = parse_schemas()

    # 2. Labels & References
    parse_labels_and_references(concepts, id_to_qname)

    # 3. Relationships
    presentation, calculation, definition = parse_relationships(concepts, id_to_qname)

    # 4. Business Rules
    business_rules = parse_business_rules(concepts)

    # Assemble JSON
    canonical_kb = {
        "meta": {
            "name": "MCA Costing Taxonomy 2019 authority model",
            "taxonomyVersion": "Costing Taxonomy 2019 Version 1.1@31-03-2019",
            "entryPoint": "in-cost-aud-2018-04-01.xsd",
            "schemaRef": "http://www.icmai.in/xbrl/2018-04-01/in-cost-aud/in-cost-aud-2018-04-01.xsd",
            "totalConcepts": len(concepts),
            "mandatoryConceptsCount": len(business_rules["mandatoryConcepts"]),
        },
        "namespaces": namespaces,
        "types": types,
        "roles": roles,
        "concepts": concepts,
        "presentation": presentation,
        "calculation": calculation,
        "definition": definition,
        "businessRules": business_rules,
    }

    # Save to file
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(canonical_kb, f, indent=2, ensure_ascii=False)



    print("=" * 60)
    print(f"DONE! File generated: {OUTPUT_JSON}")
    print(f"Size: {OUTPUT_JSON.stat().st_size / (1024*1024):.2f} MB")
    print("=" * 60)



Project root detected: /home/vanshsharma/projects/xbrl_autopilot
Taxonomy directory exists: True
Rules Excel exists: True


In [47]:
main()

Starting MCA Costing Taxonomy 2019 Extraction Pipeline
[1/4] Parsing XSD schemas...
  -> 452 concepts, 12 types, 25 roles found.
[2/4] Parsing English & Hindi labels, and Cost references...


  -> Labels and references merged into concepts.
[3/4] Parsing PRE, CAL, DEF linkbases...
  -> PRE roles: 23, CAL roles: 10, DEF roles: 20.
[4/4] Parsing Business Rules from Excel (mandatory/null checks)...
  -> 103 mandatory concepts found (cannot be null).
  -> 11 tables with mandatory line-items extracted.
DONE! File generated: /home/vanshsharma/projects/xbrl_autopilot/cost_knowledge_base.json
Size: 0.65 MB


In [48]:
"""
c.py - Simple MCA IND-AS Taxonomy 2017 & Business Rules Parser
Target: ind_as_knowledge_base.json banane ke liye simple aur readable code.
"""

import json
import re
from pathlib import Path
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook

# -------------------------------------------------------------------------
# 1. PATHS SETUP (Auto-detects project root from any folder or notebook)
# -------------------------------------------------------------------------
def get_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "IND-AS taxonomy").exists():
            return p
    try:
        f = Path(__file__).resolve()
        for p in [f.parent, *f.parents]:
            if (p / "data" / "IND-AS taxonomy").exists():
                return p
    except NameError:
        pass
    return Path("/home/vanshsharma/projects/xbrl_autopilot")

BASE_DIR = get_project_root()
INDAS_ROOT = BASE_DIR / "data" / "IND-AS taxonomy"
TAXONOMY_DIR = INDAS_ROOT / "Taxonomy for IND-AS V1.2@31-03-2017" / "Ind AS taxonomy" / "Ind AS taxonomy"
RULES_EXCEL = INDAS_ROOT / "buisness_rules" / "BusinessRulesforIndASTaxonomyV1.2@-" / "Business Rules for IndAS Taxonomy V1.2.xlsx"
OUTPUT_JSON = BASE_DIR / "ind_as_knowledge_base.json"

print(f"Project root detected: {BASE_DIR}")
print(f"Taxonomy directory exists: {TAXONOMY_DIR.exists()}")
print(f"Rules Excel exists: {RULES_EXCEL.exists()}")

# XML Namespaces
XS = "{http://www.w3.org/2001/XMLSchema}"
LINK = "{http://www.xbrl.org/2003/linkbase}"
XLINK = "{http://www.w3.org/1999/xlink}"
XBRLI = "{http://www.xbrl.org/2003/instance}"


# -------------------------------------------------------------------------
# 2. STEP 1: PARSE XSD SCHEMAS (CONCEPTS, TYPES, ROLES)
# -------------------------------------------------------------------------
def parse_schemas():
    print("[1/4] Parsing XSD schemas...")
    namespaces = {
        "in-ca": "http://www.icai.org/xbrl/taxonomy/2017-03-31/in-ca",
        "in-ca-roles": "http://www.icai.org/xbrl/taxonomy/2017-03-31/in-ca-roles",
        "in-ca-types": "http://www.icai.org/xbrl/taxonomy/2017-03-31/in-ca-types",
        "in-ci-ent": "http://www.icai.org/xbrl/taxonomy/2017-03-31/in-ca/in-ci-ent",
        "ind-as": "http://www.icai.org/xbrl/taxonomy/2017-03-31/ind-as",
        "ind-as-roles": "http://www.icai.org/xbrl/taxonomy/2017-03-31/ind-as-roles",
    }
    
    concepts = {}
    types = {}
    roles = {}
    id_to_qname = {}

    for xsd_file in sorted(TAXONOMY_DIR.rglob("*.xsd")):
        rel_path = str(xsd_file.relative_to(TAXONOMY_DIR))
        tree = ET.parse(xsd_file)
        root = tree.getroot()
        tns = root.attrib.get("targetNamespace", "")

        prefix = next((p for p, uri in namespaces.items() if uri == tns), None)
        if not prefix:
            continue

        # A. Custom Types
        for ct in root.findall(f".//{XS}complexType") + root.findall(f".//{XS}simpleType"):
            name = ct.attrib.get("name")
            restr = ct.find(f".//{XS}restriction")
            if name and restr is not None:
                enums = [e.attrib.get("value") for e in restr.findall(f"{XS}enumeration")]
                pat = restr.find(f"{XS}pattern")
                length = restr.find(f"{XS}length")
                
                t_data = {"base": restr.attrib.get("base", "xbrli:stringItemType"), "source": rel_path}
                if enums: t_data["enumerations"] = enums
                if pat is not None: t_data["pattern"] = pat.attrib.get("value")
                if length is not None: t_data["length"] = length.attrib.get("value")
                types[f"{prefix}:{name}"] = t_data

        # B. Roles (ELRs)
        for rt in root.findall(f".//{LINK}roleType"):
            uri = rt.attrib.get("roleURI")
            def_el = rt.find(f"{LINK}definition")
            def_text = def_el.text.strip() if def_el is not None and def_el.text else ""
            m = re.match(r"^\[([0-9]{6}[a-z]?)\]", def_text)
            code = m.group(1) if m else uri.split("/")[-1]
            roles[uri] = {
                "id": rt.attrib.get("id"),
                "definition": def_text,
                "code": code,
                "source": rel_path,
            }

        # C. Concepts
        for elem in root.findall(f"{XS}element"):
            name = elem.attrib.get("name")
            if not name:
                continue
            qname = f"{prefix}:{name}"
            elem_id = elem.attrib.get("id")

            concepts[qname] = {
                "qname": qname,
                "name": name,
                "prefix": prefix,
                "namespace": tns,
                "type": elem.attrib.get("type"),
                "periodType": elem.attrib.get(f"{XBRLI}periodType"),
                "balance": elem.attrib.get(f"{XBRLI}balance"),
                "abstract": elem.attrib.get("abstract") == "true",
                "nillable": elem.attrib.get("nillable") == "true",
                "source": rel_path,
                "labels": {},
                "references": [],
            }
            if elem_id:
                id_to_qname[f"{xsd_file.name}#{elem_id}"] = qname
                id_to_qname[elem_id] = qname

    print(f"  -> {len(concepts)} concepts, {len(types)} types, {len(roles)} roles found.")
    return namespaces, concepts, types, roles, id_to_qname


# -------------------------------------------------------------------------
# 3. STEP 2: PARSE LABELS & REFERENCES
# -------------------------------------------------------------------------
def parse_labels_and_references(concepts, id_to_qname):
    print("[2/4] Parsing labels and references...")

    def resolve_qn(frag):
        qn = id_to_qname.get(frag)
        if not qn and "_" in frag:
            p, _, n = frag.partition("_")
            if f"{p}:{n}" in concepts:
                qn = f"{p}:{n}"
        return qn

    # Label files (in-ca-label-2017-03-31.xml, ind-as-label-2017-03-31.xml)
    for lab_file in sorted(TAXONOMY_DIR.rglob("*lab*.xml")):
        tree = ET.parse(lab_file)
        root = tree.getroot()
        
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            href = loc.attrib.get(f"{XLINK}href", "")
            frag = href.split("#")[-1]
            qn = resolve_qn(frag)
            if qn:
                locs[lbl] = qn

        resources = {}
        for r in root.findall(f".//{LINK}label"):
            lbl = r.attrib.get(f"{XLINK}label")
            lang = r.attrib.get("{http://www.w3.org/XML/1998/namespace}lang") or r.attrib.get("xml:lang") or "en"
            role = (r.attrib.get(f"{XLINK}role") or "").split("/")[-1] or "label"
            resources[lbl] = (r.text.strip() if r.text else "", lang, role)

        for arc in root.findall(f".//{LINK}labelArc"):
            from_lbl = arc.attrib.get(f"{XLINK}from")
            to_lbl = arc.attrib.get(f"{XLINK}to")
            qn = locs.get(from_lbl)
            if qn and to_lbl in resources and qn in concepts:
                text, lang, role = resources[to_lbl]
                if lang == "hi":
                    concepts[qn]["labels"]["label_hi"] = text
                else:
                    concepts[qn]["labels"][role] = text

    # Reference files (in-ca-ref-2017-03-31.xml, ind-as-ref-2017-03-31.xml)
    for ref_file in sorted(TAXONOMY_DIR.rglob("*ref*.xml")):
        tree = ET.parse(ref_file)
        root = tree.getroot()
        locs = {}
        for loc in root.findall(f".//{LINK}loc"):
            lbl = loc.attrib.get(f"{XLINK}label")
            frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
            qn = resolve_qn(frag)
            if qn:
                locs[lbl] = qn

        ref_resources = {}
        for r in root.findall(f".//{LINK}reference"):
            lbl = r.attrib.get(f"{XLINK}label")
            parts = {child.tag.split("}")[-1]: child.text.strip() for child in r if child.text}
            if lbl and parts:
                ref_resources[lbl] = parts

        for arc in root.findall(f".//{LINK}referenceArc"):
            qn = locs.get(arc.attrib.get(f"{XLINK}from"))
            to_lbl = arc.attrib.get(f"{XLINK}to")
            if qn and to_lbl in ref_resources and qn in concepts:
                concepts[qn]["references"].append(ref_resources[to_lbl])

    # Fallback labels for typed domain containers (MCA does not provide labels in XML for domain syntax constructs)
    for qn, c in concepts.items():
        if not c["labels"].get("label"):
            if c["name"].lower().endswith("domain") or c["name"].endswith("Member"):
                c["isTypedDomain"] = True
                readable = re.sub(r'(?<!^)(?=[A-Z])', ' ', c["name"])
                if readable.lower().endswith('domain'):
                    readable = re.sub(r'\s*domain$', '', readable, flags=re.I) + ' [Domain]'
                c["labels"]["label"] = readable

    print("  -> Labels and references merged into concepts.")


# -------------------------------------------------------------------------
# 4. STEP 3: PARSE LINKBASES (PRE, CAL, DEF RELATIONSHIPS)
# -------------------------------------------------------------------------
def parse_relationships(concepts, id_to_qname):
    print("[3/4] Parsing PRE, CAL, DEF linkbases...")
    presentation = {}
    calculation = {}
    definition = {}

    def resolve_qn(frag):
        qn = id_to_qname.get(frag)
        if not qn and "_" in frag:
            p, _, n = frag.partition("_")
            if f"{p}:{n}" in concepts:
                qn = f"{p}:{n}"
        return qn

    for xml_file in sorted(TAXONOMY_DIR.rglob("*.xml")):
        if any(k in str(xml_file).lower() for k in ["-lab", "label", "-ref", "ref.", "meta-inf"]):
            continue

        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
        except Exception:
            continue

        for link in root:
            elr = link.attrib.get(f"{XLINK}role")
            if not elr:
                continue

            locs = {}
            for loc in link.findall(f"{LINK}loc"):
                lbl = loc.attrib.get(f"{XLINK}label")
                frag = loc.attrib.get(f"{XLINK}href", "").split("#")[-1]
                qn = resolve_qn(frag)
                if qn:
                    locs[lbl] = qn

            # A. Presentation Link
            for arc in link.findall(f"{LINK}presentationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    order = float(arc.attrib.get("order", "1.0"))
                    presentation.setdefault(elr, []).append({
                        "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

            # B. Calculation Link
            for arc in link.findall(f"{LINK}calculationArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    w = float(arc.attrib.get("weight", "1.0"))
                    order = float(arc.attrib.get("order", "1.0"))
                    calculation.setdefault(elr, []).append({
                        "from": f, "to": t, "weight": int(w) if w.is_integer() else w, "order": int(order) if order.is_integer() else order
                    })

            # C. Definition Link
            for arc in link.findall(f"{LINK}definitionArc"):
                f = locs.get(arc.attrib.get(f"{XLINK}from"))
                t = locs.get(arc.attrib.get(f"{XLINK}to"))
                if f and t:
                    arcrole = arc.attrib.get(f"{XLINK}arcrole", "").split("/")[-1]
                    order = float(arc.attrib.get("order", "1.0"))
                    definition.setdefault(elr, []).append({
                        "arcrole": arcrole, "from": f, "to": t, "order": int(order) if order.is_integer() else order
                    })

    for d in [presentation, calculation, definition]:
        for k in d:
            d[k].sort(key=lambda x: (x["from"], x["order"]))

    print(f"  -> PRE roles: {len(presentation)}, CAL roles: {len(calculation)}, DEF roles: {len(definition)}.")
    return presentation, calculation, definition


# -------------------------------------------------------------------------
# 5. STEP 4: PARSE BUSINESS RULES (MANDATORY & NULL CHECKS)
# -------------------------------------------------------------------------
def parse_business_rules(concepts):
    print("[4/4] Parsing Business Rules from Excel (mandatory/null checks)...")
    business_rules = {
        "mandatoryConcepts": [],      # Globally mandatory elements (cannot be null)
        "mandatoryConsolidated": [],  # Mandatory in Consolidated reports
        "conditionalMandatory": [],   # Mandatory on condition
        "mandatoryLineItems": {},     # Table -> List of mandatory fields
        "genericRules": [],
        "rules": [],
    }

    if not RULES_EXCEL.exists():
        print(f"  ! Warning: Business rules excel not found at {RULES_EXCEL}")
        return business_rules

    wb = CalamineWorkbook.from_path(RULES_EXCEL)
    sheet_names = wb.sheet_names

    by_name = {c["name"].lower(): qn for qn, c in concepts.items()}

    # A. Specific rules for elements
    spec_sheet = next((s for s in sheet_names if "specific rules" in s.lower()), None)
    if spec_sheet:
        rows = wb.get_sheet_by_name(spec_sheet).to_python()
        cur_elr = None
        for r in rows[2:]:
            c0 = str(r[0] or "").strip()
            c1 = str(r[1] or "").strip()
            c5 = str(r[5] or "").strip() if len(r) > 5 else ""

            if c0 == "LinkRole":
                cur_elr = c1
                continue
            if c0 == "Definition":
                m = re.search(r"\[([0-9]{6}[a-z]?)\]", c1)
                if m: cur_elr = m.group(1)
                continue
            if not c1 or c1 == "name" or not c5:
                continue

            status = str(r[7] or "").strip() if len(r) > 7 else ""
            first_time = str(r[8] or "").strip().upper() == "Y" if len(r) > 8 else False

            qn = by_name.get(c1.lower())
            rule_lower = c5.lower()

            if "mandatory in case of consolidated" in rule_lower:
                if qn and qn not in business_rules["mandatoryConsolidated"]:
                    business_rules["mandatoryConsolidated"].append(qn)
                    concepts[qn]["isMandatoryConsolidated"] = True
                rule_type = "mandatory-consolidated"
            elif "is a mandatory field" in rule_lower or "this is a mandatory field" in rule_lower or "mandatory field" in rule_lower:
                if qn and qn not in business_rules["mandatoryConcepts"]:
                    business_rules["mandatoryConcepts"].append(qn)
                    concepts[qn]["isMandatory"] = True
                rule_type = "mandatory"
            elif "mandatory if" in rule_lower:
                if qn and qn not in business_rules["conditionalMandatory"]:
                    business_rules["conditionalMandatory"].append(qn)
                    concepts[qn]["isMandatoryConditional"] = True
                rule_type = "conditional-mandatory"
            else:
                rule_type = "validation-rule"

            rule_obj = {
                "prefix": c0,
                "element": c1,
                "concept": qn,
                "elr": cur_elr,
                "type": rule_type,
                "ruleText": c5
            }
            if status:
                rule_obj["status"] = status
            if first_time:
                rule_obj["firstTimeAdoption"] = True

            business_rules["rules"].append(rule_obj)

    # B. Mandatory Line Items & Table Rules
    business_rules["tableRules"] = {}
    ml_sheet = next((s for s in sheet_names if "mandatory line" in s.lower()), None)
    if ml_sheet:
        ml_rows = wb.get_sheet_by_name(ml_sheet).to_python()
        for r in ml_rows[1:]:
            if len(r) < 3 or not r[1]: continue
            tbl = str(r[1] or "").strip()
            raw_text = str(r[2] or "").strip()
            if not raw_text or tbl == "Table Name": continue

            lower = raw_text.lower()
            clean_text = re.sub(r"\[.*?\]", "", raw_text)
            clean_text = re.sub(r"\(closing\s*\)", "", clean_text, flags=re.I)
            clean_text = re.sub(r"Element for '(\w+)' is mandatory\.", r"\1", clean_text)
            clean_text = re.sub(r"^\s*\d+\.\s*", "", clean_text, flags=re.M)

            tokens = re.split(r"[\n,]+|\s+[aA]nd\s+|(?<=[a-z])\s*-\s*(?=[A-Z])", clean_text)
            matched_qns = []
            for tok in tokens:
                clean_tok = tok.strip().strip("'\"-")
                clean_tok = re.sub(r"^(Mandatory elements are\s*[:-]?|Elements mandatory are\s*[:-]?|Mandatory Elements are\s*[:-]?)", "", clean_tok, flags=re.I).strip().strip("'\"-")
                if not clean_tok: continue

                cqn = by_name.get(clean_tok.lower())
                if cqn and cqn not in matched_qns:
                    matched_qns.append(cqn)
                else:
                    for w in re.findall(r"[A-Z][a-zA-Z0-9]+", clean_tok):
                        cqn_w = by_name.get(w.lower())
                        if cqn_w and cqn_w not in matched_qns:
                            matched_qns.append(cqn_w)

            if "all elements are mandatory except" in lower or "all elements mandatory except" in lower:
                business_rules["tableRules"][tbl] = {
                    "ruleType": "all_items_mandatory_except",
                    "exemptConcepts": matched_qns,
                    "rawRuleText": raw_text
                }
            elif "all elements are mandatory" in lower or "all line items are mandatory" in lower:
                business_rules["tableRules"][tbl] = {
                    "ruleType": "all_line_items_mandatory",
                    "rawRuleText": raw_text
                }
            else:
                business_rules["tableRules"][tbl] = {
                    "ruleType": "specific_mandatory_line_items",
                    "mandatoryConcepts": matched_qns,
                    "rawRuleText": raw_text
                }
                if matched_qns:
                    business_rules["mandatoryLineItems"][tbl] = matched_qns

    # C. Generic rules
    gen_sheet = next((s for s in sheet_names if "generic" in s.lower()), None)
    if gen_sheet:
        gen_rows = wb.get_sheet_by_name(gen_sheet).to_python()
        for r in gen_rows:
            if len(r) >= 2 and r[0] is not None and str(r[0]).strip().replace(".0", "").isdigit():
                business_rules["genericRules"].append({
                    "sno": int(float(r[0])),
                    "description": str(r[1]).strip()
                })

    print(f"  -> {len(business_rules['mandatoryConcepts'])} globally mandatory concepts found (cannot be null).")
    print(f"  -> {len(business_rules['mandatoryLineItems'])} tables with mandatory line-items extracted.")
    return business_rules


# -------------------------------------------------------------------------
# 6. MAIN COMPILER: RUN ALL & EXPORT JSON
# -------------------------------------------------------------------------
def main():
    print("=" * 60)
    print("Starting MCA IND-AS Taxonomy 2017 Extraction Pipeline")
    print("=" * 60)

    # 1. Schemas
    namespaces, concepts, types, roles, id_to_qname = parse_schemas()

    # 2. Labels & References
    parse_labels_and_references(concepts, id_to_qname)

    # 3. Relationships
    presentation, calculation, definition = parse_relationships(concepts, id_to_qname)

    # 4. Business Rules
    business_rules = parse_business_rules(concepts)

    # Assemble JSON
    canonical_kb = {
        "meta": {
            "name": "MCA IND-AS Taxonomy 2017 authority model",
            "taxonomyVersion": "Taxonomy for IND-AS V1.2@31-03-2017",
            "entryPoint": "in-ci-ent-2017-03-31.xsd",
            "schemaRef": "http://www.icai.org/xbrl/taxonomy/2017-03-31/in-ca/in-ci-ent-2017-03-31.xsd",
            "totalConcepts": len(concepts),
            "mandatoryConceptsCount": len(business_rules["mandatoryConcepts"]),
        },
        "namespaces": namespaces,
        "types": types,
        "roles": roles,
        "concepts": concepts,
        "presentation": presentation,
        "calculation": calculation,
        "definition": definition,
        "businessRules": business_rules,
    }

    # Save to file
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(canonical_kb, f, indent=2, ensure_ascii=False)

    print("=" * 60)
    print(f"DONE! File generated: {OUTPUT_JSON}")
    print(f"Size: {OUTPUT_JSON.stat().st_size / (1024*1024):.2f} MB")
    print("=" * 60)



Project root detected: /home/vanshsharma/projects/xbrl_autopilot
Taxonomy directory exists: True
Rules Excel exists: True


In [49]:

main()

Starting MCA IND-AS Taxonomy 2017 Extraction Pipeline
[1/4] Parsing XSD schemas...
  -> 5647 concepts, 32 types, 246 roles found.
[2/4] Parsing labels and references...
  -> Labels and references merged into concepts.
[3/4] Parsing PRE, CAL, DEF linkbases...
  -> PRE roles: 66, CAL roles: 39, DEF roles: 229.
[4/4] Parsing Business Rules from Excel (mandatory/null checks)...
  -> 220 globally mandatory concepts found (cannot be null).
  -> 96 tables with mandatory line-items extracted.
DONE! File generated: /home/vanshsharma/projects/xbrl_autopilot/ind_as_knowledge_base.json
Size: 7.54 MB
